In [1]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sklearn
print(sklearn.__version__)


1.8.0


In [ ]:
# Prepare Phase-wise Features

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


matches_df = pd.read_csv("matches.csv")
deliveries_df = pd.read_csv("deliveries.csv")



matches_df['winner'] = matches_df['winner'].fillna('No Result')
merged_df = deliveries_df.merge(matches_df, left_on='match_id', right_on='id')

def get_phase(over):
    match over:
        case o if o < 6:
            return 'Powerplay'
        case o if o < 15:
            return 'Middle'
        
        case _:
            return 'Death'
        


    

merged_df['phase'] = merged_df['over'].apply(get_phase)

print(merged_df[['match_id', 'batting_team', 'phase', 'total_runs']].head(10))

   match_id           batting_team      phase  total_runs
0    335982  Kolkata Knight Riders  Powerplay           1
1    335982  Kolkata Knight Riders  Powerplay           0
2    335982  Kolkata Knight Riders  Powerplay           1
3    335982  Kolkata Knight Riders  Powerplay           0
4    335982  Kolkata Knight Riders  Powerplay           0
5    335982  Kolkata Knight Riders  Powerplay           0
6    335982  Kolkata Knight Riders  Powerplay           1
7    335982  Kolkata Knight Riders  Powerplay           0
8    335982  Kolkata Knight Riders  Powerplay           4
9    335982  Kolkata Knight Riders  Powerplay           4


In [4]:
#  Create Feature Table
phase_runs = merged_df.groupby(['match_id', 'batting_team', 'phase'])['total_runs'].sum().unstack(fill_value=0).reset_index()
phase_runs.columns = ['Match ID', 'Team', 'Death Runs', 'Middle Runs', 'Powerplay Runs']

phase_runs['Won'] = (phase_runs.apply(
    lambda r: matches_df[matches_df['id'] == r['Match ID']]['winner'].values[0] == r['Team'], axis=1)
).astype(int)


print(phase_runs.head(10))
print("\nShape:", phase_runs.shape)



   Match ID                         Team  Death Runs  Middle Runs  \
0    335982        Kolkata Knight Riders          68           93   
1    335982  Royal Challengers Bangalore           1           55   
2    335983          Chennai Super Kings          79          108   
3    335983              Kings XI Punjab          42          102   
4    335984             Delhi Daredevils           4           73   
5    335984             Rajasthan Royals          33           56   
6    335985               Mumbai Indians          60           58   
7    335985  Royal Challengers Bangalore          48           78   
8    335986              Deccan Chargers          33           38   
9    335986        Kolkata Knight Riders          31           55   

   Powerplay Runs  Won  
0              61    1  
1              26    0  
2              53    1  
3              63    0  
4              55    1  
5              40    0  
6              47    0  
7              40    1  
8              

In [5]:
#  Split Data into Train & Test
X = phase_runs[['Powerplay Runs', 'Middle Runs', 'Death Runs']]
y = phase_runs['Won']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 1749
Testing samples: 438


In [6]:
# Train the Model 

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [7]:
# Evaluate the Model 

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy*100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Loss', 'Win']))

Model Accuracy: 60.27%

Classification Report:
              precision    recall  f1-score   support

        Loss       0.60      0.59      0.60       216
         Win       0.61      0.61      0.61       222

    accuracy                           0.60       438
   macro avg       0.60      0.60      0.60       438
weighted avg       0.60      0.60      0.60       438



In [8]:
# Add More Features

phase_runs['Total Runs'] = phase_runs['Powerplay Runs'] + phase_runs['Middle Runs'] + phase_runs['Death Runs']
phase_runs['Death %'] = (phase_runs['Death Runs'] / phase_runs['Total Runs'] * 100).round(2)
phase_runs['Powerplay %'] = (phase_runs['Powerplay Runs'] / phase_runs['Total Runs'] * 100).round(2)
phase_runs['Middle %'] = (phase_runs['Middle Runs'] / phase_runs['Total Runs'] * 100).round(2)

inning_df = merged_df.groupby(['match_id', 'batting_team'])['inning'].first().reset_index()
inning_df.columns = ['Match ID', 'Team', 'Inning']

phase_runs = phase_runs.merge(inning_df, on=['Match ID', 'Team'])

wickets = merged_df.groupby(['match_id', 'batting_team'])['is_wicket'].sum().reset_index()
wickets.columns = ['Match ID', 'Team', 'Wickets Lost']
phase_runs = phase_runs.merge(wickets, on=['Match ID', 'Team'])

X = phase_runs[['Powerplay Runs', 'Middle Runs', 'Death Runs',
                'Total Runs', 'Death %', 'Powerplay %', 'Middle %',
                'Inning', 'Wickets Lost']]
y = phase_runs['Won']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)



y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy*100:.2f}%")
print("\nClassification Report:")

print(classification_report(y_test, y_pred, target_names=['Loss', 'Win']))

Model Accuracy: 70.09%

Classification Report:
              precision    recall  f1-score   support

        Loss       0.69      0.72      0.70       216
         Win       0.71      0.68      0.70       222

    accuracy                           0.70       438
   macro avg       0.70      0.70      0.70       438
weighted avg       0.70      0.70      0.70       438

